### Creating View

In [0]:
select current_metastore();

In [0]:
%python

for dirs in dbutils.fs.ls('/Volumes/workspace/default/kirti_volume'):
    print(dirs.path)
    dbutils.fs.rm(dirs.path, True)

## Unity Catalog 
### Unity Catalog is a unified governance solution for data and AI assets in Databricks. Think of it as a central system that helps you organize, secure, and manage all your data—like tables, files, machine learning models, and more—across your Databricks workspaces.

In [0]:
-- create catalog
CREATE CATALOG IF NOT EXISTS P10AT_catalog;

-- set this catalog as defualt catalog
USE CATALOG P10AT_catalog;

-- what is defualt catalog currently
SELECT current_catalog();

-- create schema
CREATE SCHEMA IF NOT EXISTS P10AT_catalog.kirti_db;

-- set this schema as default schema
USE SCHEMA kirti_db;

SELECT current_schema();

In [0]:
SELECT current_catalog();

In [0]:

CREATE or REPLACE TABLE p10at_catalog.kirti_db.sales(s_id INT, 
                    product_id STRING, 
                    product_category STRING,
                    emp_id INT,
                    cust_id INT,
                    amount FLOAT);

INSERT INTO sales VALUES(1, 'P100', 'office', 101, 1001, 10000),
                        (2, 'P100', 'office', 101, 1002, 11000),
                        (3, 'P100', 'office', 101, 1003, 12000),
                        (4, 'P101', 'home', 102, 1001, 11500),
                        (5, 'P101', 'home', 102, 1001, 12500),
                        (6, 'P101', 'home', 102, 1001, 10500),
                        (7, 'P102', 'electronics', 103, 1001, 11100),
                        (8, 'P102', 'electronics', 103, 1001, 12100),
                        (9, 'P102', 'electronics', 103, 1001, 13100),
                        (10, 'P103', 'electronics', 104, 1001, 15000)

In [0]:
SELECT * FROM sales;

In [0]:
SHOW TABLES;

In [0]:
--A view is a logical table created on top of a query.
--It does NOT store data, only the SQL definition.
CREATE OR REPLACE VIEW electonics_products 
AS
SELECT * FROM sales WHERE product_category = 'electronics';

In [0]:
SHOW TABLES;

In [0]:
--A temporary view is similar to a view but exists only for the current Spark session.
CREATE OR REPLACE TEMPORARY VIEW  office
AS
SELECT * FROM sales WHERE product_category = 'office';

In [0]:
SHOW TABLES

In [0]:
-- Materialized views can be created only with the Unity catalog enabled workspace
/*What is a materialized view?
A materialized view is a database object that stores the results of a query as a physical table. 
 their data from the underlying tables, materialized views contain precomputed data that is incrementally 
 updated on a schedule or on demand. This precomputation of data allows for faster query response times
  and improved performance in certain scenarios.*/
CREATE OR REPLACE MATERIALIZED VIEW sales_summary
AS
SELECT product_category, sum(amount) as TOTAL_SALES, avg(amount) as AVG_SALES
    FROM P10AT_catalog.kirti_db.sales
    GROUP BY product_category;

### Python Syntax: createOrReplaceTempView

In [0]:
%python

sales_schema = ["s_id", "product_id", "product_category", "emp_id", "cust_id", "amount"]

sales_data = [
                (1, 'P100', 'office', 101, 1001, 10000),
                (2, 'P100', 'office', 101, 1002, 11000),
                (3, 'P100', 'office', 101, 1003, 12000),
                (4, 'P101', 'home', 102, 1001, 11500),
                (5, 'P101', 'home', 102, 1001, 12500),
                (6, 'P101', 'home', 102, 1001, 10500),
                (7, 'P102', 'electronics', 103, 1001, 11100),
                (8, 'P102', 'electronics', 103, 1001, 12100),
                (9, 'P102', 'electronics', 103, 1001, 13100),
                (10, 'P103', 'electronics', 104, 1001, 15000)
]
 
salesDF = spark.createDataFrame(data=sales_data, schema=sales_schema)

salesDF.show()

In [0]:
%python

# temp table or temp view created within spark session
# create temp table out of data frame, which can be used in spark sql

salesDF.createOrReplaceTempView("salesview")

In [0]:
DESCRIBE salesview;

In [0]:
SHOW TABLES;

In [0]:
%python
spark.sql("SELECT * FROM salesview").show()


========================================== END =====================================